# SC-Mamba: Multivariate Benchmark — 17 Datasets (Direction 3)
**Mục tiêu:** Huấn luyện độc lập 17 datasets với Cross-Asset Graph (`num_assets > 1`), auto-derive cấu hình tối ưu cho từng dataset, thu thập Benchmark Bảng Lớn.
- Script: `core/train.py` (đúng, không phải `main.py`)
- Config: `core/config.v_multivariate_{dataset}.yaml` (tạo tự động mỗi lần lặp)
- Checkpoints: `/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints/`

## Cell 1 — Environment Setup (identical to 00_Train_SCMamba.ipynb)

In [1]:
# Uninstall conflicting Colab defaults
!pip uninstall -y transformers sentence-transformers torch torchvision torchaudio

# Install pinned PyTorch cu121 (must match Mamba wheel builds)
!pip install torch==2.4.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install transformers==4.39.3 packaging triton==3.0.0

# Pre-built Mamba2 wheels (saves 20-30 min vs compiling from source)
!wget -qO causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl "https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.4.0/causal_conv1d-1.4.0%2Bcu122torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!wget -qO mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl "https://github.com/state-spaces/mamba/releases/download/v2.2.4/mamba_ssm-2.2.4%2Bcu12torch2.4cxx11abiFALSE-cp312-cp312-linux_x86_64.whl"
!pip install causal_conv1d-1.4.0-cp312-cp312-linux_x86_64.whl
!pip install mamba_ssm-2.2.4-cp312-cp312-linux_x86_64.whl

# Scientific stack
!pip install yfinance gluonts tqdm utilsforecast pyyaml pandas numpy submitit torchmetrics gpytorch statsforecast properscoring

Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
Found existing installation: sentence-transformers 5.2.3
Uninstalling sentence-transformers-5.2.3:
  Successfully uninstalled sentence-transformers-5.2.3
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 799.0/799.0 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 110.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 51.3 MB/s eta 0:00:00
     ━

## Cell 2 — Mount Drive & Clone Repo

In [2]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ngngsonan/SC-Mamba.git /content/SC-Mamba 2>/dev/null || echo 'Repo already cloned'
!cd /content/SC-Mamba && git pull


Mounted at /content/drive
Already up to date.


## Cell 3 — Project Root & Sanity Checks

In [3]:
import os

# =====================================================================
# REQUIRED ENVIRONMENT VARIABLES — set before any subprocess or import
# =====================================================================

# 1. TRITON_F32_DEFAULT: Prevents Triton 3.x TF32 crash on T4 GPU (Turing arch).
#    Root cause: Triton compiles a TF32 kernel path that triggers 'IndexError: map::at'
#    on Turing GPUs. Setting 'ieee' forces FP32 IEEE path which is safe.
#    Must be set BEFORE mamba-ssm or triton is imported.
#    Also set inside train.py itself, but setting here ensures subprocess inherits it.
os.environ['TRITON_F32_DEFAULT'] = 'ieee'

# 2. SC_MAMBA_DIAG: Enables diagnostic prints in models.py (padding waste, spectral stats).
#    train.py also sets this when diag_prints=True, but set here so it's active
#    if any imports are done directly in the notebook context.
os.environ['SC_MAMBA_DIAG'] = '1'

# 3. Pass env vars to subprocesses explicitly (subprocess doesn't always inherit Colab env)
SUBPROCESS_ENV = os.environ.copy()   # Pass to subprocess.run(env=SUBPROCESS_ENV)

print('✅ Environment variables set:')
print(f"   TRITON_F32_DEFAULT = {os.environ.get('TRITON_F32_DEFAULT')}")
print(f"   SC_MAMBA_DIAG      = {os.environ.get('SC_MAMBA_DIAG')}")

print()
# Auto-detect project root (mirrors 00_Train_SCMamba.ipynb logic)
if os.path.exists('/content/SC-Mamba/core/train.py'):
    PROJECT_ROOT = '/content/SC-Mamba'
elif os.path.exists(os.path.join(os.getcwd(), 'core', 'train.py')):
    PROJECT_ROOT = os.getcwd()
# # else:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'CWD          = {os.getcwd()}')

# Sanity checks
CHECKPOINT_DIR = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_checkpoints'
REAL_VAL_DIR   = os.path.join(PROJECT_ROOT, 'data', 'real_val_datasets')
LOG_DIR        = os.path.join(PROJECT_ROOT, 'logs')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(REAL_VAL_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

checks = [
    ('core/train.py',                     'Training script         '),
    ('core/eval_real_dataset.py',          'Eval script             '),
    ('core/real_data_args.yaml',           'Eval args yaml          '),
    ('data/data_provider/data_loader.py',  'Data loader             '),
    ('data/scripts/store_real_datasets.py','Dataset generator       '),
    ('data/data_provider/multivariate_loader.py', 'Multivariate loader'),
]
all_ok = True
for path, label in checks:
    exists = os.path.exists(path)
    print(f'  {"✅" if exists else "❌"} {label} → {path}')
    if not exists: all_ok = False

assert all_ok, f'❌ Missing files! CWD={os.getcwd()}'
print('\n✅ All paths verified. Ready for benchmark loop.')

✅ Environment variables set:
   TRITON_F32_DEFAULT = ieee
   SC_MAMBA_DIAG      = 1

PROJECT_ROOT = /content/SC-Mamba
CWD          = /content/SC-Mamba
  ✅ Training script          → core/train.py
  ✅ Eval script              → core/eval_real_dataset.py
  ✅ Eval args yaml           → core/real_data_args.yaml
  ✅ Data loader              → data/data_provider/data_loader.py
  ✅ Dataset generator        → data/scripts/store_real_datasets.py
  ✅ Multivariate loader → data/data_provider/multivariate_loader.py

✅ All paths verified. Ready for benchmark loop.


## Cell 4 — BASE_CONFIG (SSM Architecture — DO NOT CHANGE)
Base config mirrors `00_Train_SCMamba.ipynb` architecture exactly.
Dataset-specific params (`num_assets`, `pred_len`, `context_len`) are **auto-overridden** in the loop.

In [10]:
# ⬇️  USER SETTINGS — edit only this block
MODEL_VER         = 'vmul'          # Prefix for all checkpoint names
CHECKPOINT_DIR    = '/content/drive/MyDrive/Colab Notebooks/SCMamba/sc_mamba_multivariate_ckp'
NUM_EPOCHS        = 4 #30                         # Per-dataset epochs
TRAINING_ROUNDS   = 20 #300                        # Batches per epoch
BATCH_SIZE        = 4                          # Small: each item = N aligned series
PRIOR_MIX_FRAC    = 0.3                        # 30% synthetic GP, 70% real multivariate
BETA_KL           = 0.2
CONTINUE_TRAINING = False                      # False = train from scratch each dataset
VALIDATION_ROUNDS = 5 #100
REAL_TEST_INTERVAL = 5

# ─── RESUME SETTINGS (for Colab disconnects) ───────────────────────
# RESUME_FROM: Start loop from this dataset key (skip earlier ones).
#   None  = start from beginning (default)
#   'cif_2016' = resume from cif_2016, skipping exchange_rate etc.
RESUME_FROM = None   # e.g. 'cif_2016'

# SKIP_COMPLETED: Auto-detect existing checkpoints and skip those datasets.
#   True  = if SCMamba_v_multivariate_{key}_best.pth exists → skip
#   False = re-train even if checkpoint exists (use to force re-train)
SKIP_COMPLETED = False

# AUTO_CONTINUE: If checkpoint exists, set continue_training=True automatically.
#   True  (recommended) = resume epoch from checkpoint, not from 0
#   False = ignore existing checkpoint, train fresh
AUTO_CONTINUE = True

# SSM architecture (identical to 00_Train_SCMamba.ipynb — do not change)
BASE_CONFIG = {
    'seed': 42,
    'debugging': False,
    'scaler': 'min_max',
    'sin_pos_enc': False,
    'sin_pos_const': False,
    'sub_day': True,
    'encoding_dropout': 0.1,
    'handle_constants_model': True,
    'diag_prints': True,

    # Mamba2 SSM backbone — mirrors 00_Train_SCMamba.ipynb exactly
    'ssm_config': {
        'bidirectional': False,
        'enc_conv': True,
        'init_dil_conv': True,
        'enc_conv_kernel': 5,
        'init_conv_kernel': 5,
        'init_conv_max_dilation': 3,
        'global_residual': False,
        'in_proj_norm': False,
        'initial_gelu_flag': True,
        'linear_seq': 15,
        'mamba2': True,
        'norm': True,
        'norm_type': 'layernorm',
        'num_encoder_layers': 2,
        'd_state': 128,
        'residual': False,
        'token_embed_len': 1024,
        'block_expansion': 2,
        'chunk_size': 256,      # adaptive_chunk_size() in train.py will auto-lower this
        'headdim': 128,         # CRITICAL: must be >= 128 to avoid Triton LLVM crash on T4
    },

    'lr_scheduler': 'cosine',
    'initial_lr': 5e-5,
    'learning_rate': 1e-7,
    't_max': -1,
    'multipoint': True,
    'pred_len_sample': False,   # Use fixed pred_len for proper benchmarking
    'sample_multi_pred': 0.0,
    'loss': 'nll',
    'wandb': False,
    'no_pos_enc': False,
    'real_test_interval': REAL_TEST_INTERVAL,    # Run validation eval every 5 epochs

    'prior_config': {
        'prior_mix_frac': PRIOR_MIX_FRAC,
        'curriculum_learning': False,
        'mixup_prob': 0.0,
        'mixup_series': 4,
        'damp_and_spike': True,
        'damping_noise_ratio': 0.05,
        'spike_noise_ratio': 0.05,
        'spike_signal_ratio': 0.05,
        'spike_batch_ratio': 0.05,
        'fp_options': {
            'linear_random_walk_frac': 0,
            'seasonal_only': False,
            'scale_noise': [0.6, 0.3],
            'trend_exp': False,
            'harmonic_scale_ratio': 0.4,
            'harmonic_rate': 0.75,
            'trend_additional': True,
            'transition_ratio': 0.0,
        },
        'gp_prior_config': {
            'max_kernels': 6,
            'likelihood_noise_level': 0.4,
            'noise_level': 'random',
            'use_original_gp': False,
            'gaussians_periodic': True,
            'peak_spike_ratio': 0.1,
            'subfreq_ratio': 0.2,
            'periods_per_freq': 0.5,
            'gaussian_sampling_ratio': 0.2,
            'kernel_periods': [4, 5, 7, 21, 24, 30, 60, 120],
            'max_period_ratio': 1.0,
            'kernel_bank': {
                'matern_kernel': 1.5,
                'linear_kernel': 1,
                'periodic_kernel': 5,
                'polynomial_kernel': 0,
                'spectral_mixture_kernel': 0,
            },
        },
    },
}

print('✅ BASE_CONFIG defined.')
print(f'   mamba2={BASE_CONFIG["ssm_config"]["mamba2"]} | headdim={BASE_CONFIG["ssm_config"]["headdim"]} | loss={BASE_CONFIG["loss"]}')

✅ BASE_CONFIG defined.
   mamba2=True | headdim=128 | loss=nll


## Cell 5 — Dataset Registry & Config Generator

In [11]:
import os, yaml, copy

# =====================================================================
# DATASET REGISTRY — 17 Mamba4Cast benchmark datasets
# (key, pred_len, freq, num_assets, display_name)
# Note: use exact GluonTS keys (e.g. 'nn5_daily_without_missing' not 'nn5_daily')
# =====================================================================
DATASET_REGISTRY = [
    ('car_parts_without_missing',  12, 'M',    2658, 'Car Parts'),
    ('cif_2016',                   12, 'M',      72, 'CIF 2016'),
    ('covid_deaths',               30, 'D',     266, 'Covid Deaths'),
    ('ercot',                      24, 'H',       8, 'ERCOT Load'),
    ('exchange_rate',              30, 'B',       8, 'Exchange Rate'),
    ('fred_md',                    12, 'M',     107, 'FRED-MD'),
    ('hospital',                   12, 'M',     767, 'Hospital'),
    ('m1_monthly',                 18, 'M',     617, 'M1 Monthly'),
    ('m1_quarterly',                8, 'Q',     203, 'M1 Quarterly'),
    ('m3_monthly',                 18, 'M',    1428, 'M3 Monthly'),
    ('m3_quarterly',                8, 'Q',     756, 'M3 Quarterly'),
    ('nn5_daily_without_missing',  56, 'D',     111, 'NN5 Daily'),
    ('nn5_weekly',                  8, 'W',     111, 'NN5 Weekly'),
    ('tourism_monthly',            24, 'M',     366, 'Tourism Monthly'),
    ('tourism_quarterly',           8, 'Q',     427, 'Tourism Quarterly'),
    ('traffic',                    24, 'H',     862, 'Traffic'),
    ('weather',                    30, 'D',      21, 'Weather'),
]

# VRAM budget: cap N_assets to avoid OOM on T4 (15GB)
# SC-Mamba with adaptive_chunk_size handles short series efficiently,
# but batch_size * N * context_len must fit in VRAM.
MAX_N_ASSETS = 72   # Conservative cap; increase if GPU allows

def build_dataset_config(key, pred_len, num_assets, project_root, checkpoint_dir):
    """Create a per-dataset config dict from BASE_CONFIG + dataset-specific overrides."""
    cfg = copy.deepcopy(BASE_CONFIG)

    # ── Dataset-specific overrides ──
    cfg['real_train_datasets'] = [key]
    cfg['real_test_datasets']  = [key]     # Monitor same dataset

    # Smart context: 2x pred_len, floored to 8, capped at 256
    ctx = max(8, min(pred_len * 2, 256))
    cfg['context_len'] = ctx
    cfg['min_seq_len'] = max(8, pred_len)
    cfg['max_seq_len'] = ctx
    cfg['pred_len']     = pred_len
    cfg['pred_len_min'] = max(4, pred_len // 2)

    # Cap N for VRAM budget
    n = min(num_assets, MAX_N_ASSETS)
    cfg['num_assets'] = n

    # Batch size: smaller for large N to avoid OOM
    cfg['batch_size'] = 2 if n > 32 else 4

    cfg['num_epochs']       = NUM_EPOCHS
    cfg['training_rounds']  = TRAINING_ROUNDS
    cfg['validation_rounds'] = VALIDATION_ROUNDS
    cfg['beta_kl']           = BETA_KL
    cfg['continue_training'] = CONTINUE_TRAINING
    cfg['beta_anneal_epochs'] = max(5, NUM_EPOCHS // 5)
    cfg['model_prefix']      = checkpoint_dir
    cfg['version']           = f'{MODEL_VER}_{key}'

    # Prior: more real data for multivariate training
    cfg['prior_config']['prior_mix_frac'] = PRIOR_MIX_FRAC

    return cfg

print(f'Dataset registry loaded: {len(DATASET_REGISTRY)} datasets.')
for key, pred, freq, n, name in DATASET_REGISTRY:
    n_eff = min(n, MAX_N_ASSETS)
    ctx = max(8, min(pred * 2, 256))
    bs  = 2 if n_eff > 32 else 4
    print(f'  {name:25s}: pred={pred:3d}, N={n_eff:4d} (native={n:5d}), ctx={ctx:3d}, batch={bs}')

Dataset registry loaded: 17 datasets.
  Car Parts                : pred= 12, N=  72 (native= 2658), ctx= 24, batch=2
  CIF 2016                 : pred= 12, N=  72 (native=   72), ctx= 24, batch=2
  Covid Deaths             : pred= 30, N=  72 (native=  266), ctx= 60, batch=2
  ERCOT Load               : pred= 24, N=   8 (native=    8), ctx= 48, batch=4
  Exchange Rate            : pred= 30, N=   8 (native=    8), ctx= 60, batch=4
  FRED-MD                  : pred= 12, N=  72 (native=  107), ctx= 24, batch=2
  Hospital                 : pred= 12, N=  72 (native=  767), ctx= 24, batch=2
  M1 Monthly               : pred= 18, N=  72 (native=  617), ctx= 36, batch=2
  M1 Quarterly             : pred=  8, N=  72 (native=  203), ctx= 16, batch=2
  M3 Monthly               : pred= 18, N=  72 (native= 1428), ctx= 36, batch=2
  M3 Quarterly             : pred=  8, N=  72 (native=  756), ctx= 16, batch=2
  NN5 Daily                : pred= 56, N=  72 (native=  111), ctx=112, batch=2
  NN5 Weekly  

## Cell 6 — Download Datasets (if not already cached)

In [12]:
# Download all GluonTS PKL files needed for training
!python data/scripts/store_real_datasets.py
print('\n✅ Dataset download complete.')

  ✅ M3C.xls already exists at /root/.gluonts/datasets/M3C.xls
  📥 Downloading M1C.xls from GitHub mirror...
  ⚠️  Failed to download M1C.xls: HTTP Error 404: Not Found
      Manual fix: download from https://raw.githubusercontent.com/Mcompetitions/M1-methods/master/M1C.xls
      and place at: /root/.gluonts/datasets/M1C.xls
Dataset nn5_daily_without_missing already exists. Skipping...
Dataset nn5_weekly already exists. Skipping...
Dataset covid_deaths already exists. Skipping...
Dataset weather already exists. Skipping...
Dataset hospital already exists. Skipping...
Dataset fred_md already exists. Skipping...
Dataset car_parts_without_missing already exists. Skipping...
Dataset traffic already exists. Skipping...
Dataset m3_monthly already exists. Skipping...
Dataset ercot already exists. Skipping...
Dataset m1_monthly already exists. Skipping...
Dataset m1_quarterly already exists. Skipping...
Dataset cif_2016 already exists. Skipping...
Dataset exchange_rate already exists. Skipping.

## Cell 7 — Main Benchmark Loop
Calls `python core/train.py --config_file core/config.v_multivariate_{dataset}.yaml`
for each dataset. Exceptions do NOT kill the loop.

In [14]:
import os, yaml, subprocess, copy

# ── ⚙️ OPTIONAL: Run only a subset for quick testing ──
SELECTED = 'all'   # 'all' | list of dataset keys e.g. ['exchange_rate', 'cif_2016']

if SELECTED == 'all':
    run_list = DATASET_REGISTRY
# # else:
    run_list = [d for d in DATASET_REGISTRY if d[0] in SELECTED]

# ─── Apply RESUME_FROM: skip datasets before the resume point ───
if RESUME_FROM is not None:
    keys = [d[0] for d in run_list]
    if RESUME_FROM in keys:
        start_idx = keys.index(RESUME_FROM)
        skipped = [d[4] for d in run_list[:start_idx]]
        run_list = run_list[start_idx:]
        print(f'⏩ RESUME_FROM={RESUME_FROM}: skipping {skipped}')
    # # else:
        print(f'⚠️  RESUME_FROM key "{RESUME_FROM}" not found in run_list — running all')
# # else:
BENCHMARK_LOG = os.path.join(LOG_DIR, 'log_multivariate_benchmark.txt')

with open(BENCHMARK_LOG, 'a') as f:
  f.write('\n\n================  CROSS-ASSET BENCHMARK START  ================\n')
  results_summary = []

  for key, pred_len, freq, num_assets, name in run_list:
      print(f'\n{"="*60}')
      print(f'🚀 CROSS-ASSET TRAINING: {name.upper()}')
      n_eff = min(num_assets, MAX_N_ASSETS)
      ctx   = max(8, min(pred_len * 2, 256))
      print(f'   num_assets    : {n_eff}')
      print(f'   pred_len      : {pred_len}')
      print(f'   context_len   : {ctx}')
      print(f'   num_epochs    : {NUM_EPOCHS}')
      print(f'{"="*60}')

      try:
          # 1. Build config
          cfg = build_dataset_config(key, pred_len, num_assets, PROJECT_ROOT, CHECKPOINT_DIR)

          # 2. Save config to core/
          config_name = f'config.{MODEL_VER}_{key}.yaml'
          config_path = os.path.join(PROJECT_ROOT, 'core', config_name)
          with open(config_path, 'w') as f:
              yaml.dump(cfg, f, default_flow_style=False)
          print(f'📝 Config saved: {config_path}')

          # 3. Run training — CORRECT ENTRY POINT: core/train.py
          cmd = ['python', 'core/train.py', '-c', config_path]
          log_file = os.path.join(LOG_DIR, f'log_{key}.txt')
          with open(log_file, 'w') as logf:
              result = subprocess.run(
                  cmd,
                  cwd=PROJECT_ROOT,
                  stdout=logf,
                  stderr=subprocess.STDOUT,
                  text=True
              )

          if result.returncode != 0:
              raise RuntimeError(f'Training exited with code {result.returncode}. See {log_file}')

          # 4. Extract & save metrics
          with open(log_file, 'r') as f:
              lines = f.readlines()
          tail = ''.join(lines[-20:])

          with open(BENCHMARK_LOG, 'a') as f:
              f.write(f'\n--- {name} (N={n_eff}, pred={pred_len}) ---\n')
              f.write(tail)

          results_summary.append((name, n_eff, '✅ OK'))
          print(f'✅ {name} complete → {log_file}')

      except Exception as ex:
          err_msg = str(ex)[:300]
          with open(BENCHMARK_LOG, 'a') as f:
              f.write(f'\n--- {name} FAILED: {err_msg} ---\n')
          results_summary.append((name, n_eff, f'❌ {err_msg[:60]}'))
          print(f'❌ {name} FAILED: {err_msg}')
          continue  # ← vòng lặp KHÔNG CHẾT, chạy tiếp dataset kế tiếp

  print(f'\n{"="*60}')
  print('🎉 BENCHMARK HOÀN TẤT!')
  print(f'{"="*60}')
  for name, n, status in results_summary:
      print(f'  {status:6s}  {name:25s} (N={n})')
  print(f'\nKết quả đầy đủ: {BENCHMARK_LOG}')


🚀 CROSS-ASSET TRAINING: CAR PARTS
   num_assets    : 72
   pred_len      : 12
   context_len   : 24
   num_epochs    : 4
📝 Config saved: /content/SC-Mamba/core/config.vmul_car_parts_without_missing.yaml
❌ Car Parts FAILED: Training exited with code 1. See /content/SC-Mamba/logs/log_car_parts_without_missing.txt

🚀 CROSS-ASSET TRAINING: CIF 2016
   num_assets    : 72
   pred_len      : 12
   context_len   : 24
   num_epochs    : 4
📝 Config saved: /content/SC-Mamba/core/config.vmul_cif_2016.yaml
❌ CIF 2016 FAILED: Training exited with code 1. See /content/SC-Mamba/logs/log_cif_2016.txt

🚀 CROSS-ASSET TRAINING: COVID DEATHS
   num_assets    : 72
   pred_len      : 30
   context_len   : 60
   num_epochs    : 4
📝 Config saved: /content/SC-Mamba/core/config.vmul_covid_deaths.yaml


KeyboardInterrupt: 